# 02 — Feature Engineering
**Forest Carbon Stock Estimation — Nainital District, Uttarakhand**

This notebook covers:
- Computing all five spectral indices from Sentinel-2 bands
- Deriving terrain features (slope, aspect) from SRTM DEM
- Visualising index maps
- Feature distribution analysis
- Identifying redundant features via VIF

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from pathlib import Path

from src.features.indices import (
    compute_all_indices, read_sentinel_stack,
    compute_terrain_features, BAND_ORDER
)
from src.utils.raster_utils import raster_to_dataframe

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)
PATHS = CFG['paths']

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})
print('Setup complete.')

## 1. Load Sentinel-2 Stack

In [ ]:
s2_path = Path('..') / PATHS['sentinel_stack']
bands, profile = read_sentinel_stack(s2_path)
print('Bands loaded:', list(bands.keys()))
print('Shape:', list(bands.values())[0].shape)
print('CRS:', profile['crs'])
print('Transform:', profile['transform'])

## 2. Compute Spectral Indices

In [ ]:
indices = compute_all_indices(
    B2=bands['B2'], B3=bands['B3'], B4=bands['B4'],
    B8=bands['B8'], B11=bands['B11'], B12=bands['B12']
)

print('Index statistics:')
for name, arr in indices.items():
    valid = arr[~np.isnan(arr)]
    print(f'  {name:6s}  min={valid.min():.3f}  mean={valid.mean():.3f}  max={valid.max():.3f}')

In [ ]:
# Visualise all five indices
cmaps = {'NDVI': 'RdYlGn', 'EVI': 'YlGn', 'NDMI': 'RdBu', 'NBR': 'BrBG', 'SAVI': 'YlGn'}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, (name, arr) in enumerate(indices.items()):
    p2, p98 = np.nanpercentile(arr, [2, 98])
    im = axes[i].imshow(arr, cmap=cmaps[name], vmin=p2, vmax=p98, interpolation='bilinear')
    plt.colorbar(im, ax=axes[i], fraction=0.04)
    axes[i].set_title(name, fontsize=12, fontweight='bold')
    axes[i].axis('off')

# NIR false colour in last panel
nir = bands['B8']; red = bands['B4']; grn = bands['B3']
def norm(b): p2,p98=np.nanpercentile(b,[2,98]); return np.clip((b-p2)/(p98-p2+1e-9),0,1)
rgb = np.dstack([norm(nir), norm(red), norm(grn)])
axes[5].imshow(rgb); axes[5].set_title('NIR False Colour (B8-B4-B3)', fontsize=12); axes[5].axis('off')

fig.suptitle('Sentinel-2 Spectral Indices — Nainital District', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Terrain Features

In [ ]:
dem_path = Path('..') / PATHS['dem_tif']
terrain, dem_profile = compute_terrain_features(dem_path)

print('Terrain features computed:')
for name, arr in terrain.items():
    valid = arr[~np.isnan(arr)]
    print(f'  {name:12s}  min={valid.min():.1f}  mean={valid.mean():.1f}  max={valid.max():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
terrain_cmaps = {'elevation': 'terrain', 'slope': 'YlOrRd', 'aspect': 'hsv'}

for ax, (name, arr) in zip(axes, terrain.items()):
    p2, p98 = np.nanpercentile(arr[~np.isnan(arr)], [2, 98])
    im = ax.imshow(arr, cmap=terrain_cmaps[name], vmin=p2, vmax=p98, interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.04)
    ax.set_title(name.capitalize(), fontsize=12, fontweight='bold')
    ax.axis('off')

fig.suptitle('SRTM Terrain Features — Nainital District', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Training Dataset Feature Distributions

In [ ]:
df = pd.read_csv(Path('..') / PATHS['training_csv'])
features = CFG['features']['model_features']

n = len(features)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.ravel()

for i, feat in enumerate(features):
    axes[i].hist(df[feat].dropna(), bins=40, color='#4caf50', edgecolor='white', lw=0.4)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Count', fontsize=8)
    axes[i].spines[['top','right']].set_visible(False)
    axes[i].tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions (Training Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Variance Inflation Factor (VIF) — Multicollinearity Check

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[features].dropna()
vif_data = pd.DataFrame()
vif_data['Feature'] = features
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(features))]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

print('Variance Inflation Factors (VIF > 10 = high multicollinearity):')
print(vif_data.to_string(index=False))

# Visual
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e53935' if v > 10 else '#fb8c00' if v > 5 else '#43a047'
          for v in vif_data['VIF']]
ax.barh(vif_data['Feature'], vif_data['VIF'], color=colors)
ax.axvline(10, color='red', ls='--', lw=1.2, label='VIF=10 threshold')
ax.axvline(5,  color='orange', ls='--', lw=1.0, label='VIF=5 caution')
ax.set_xlabel('VIF', fontsize=11)
ax.set_title('Variance Inflation Factors', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. NDVI Forest Mask
Pixels with NDVI > 0.3 are classified as forest and included in the prediction grid.

In [ ]:
ndvi_thresh = CFG['seq_potential']['non_forest_ndvi']
ndvi_arr = indices['NDVI']
forest_mask = ndvi_arr > ndvi_thresh

total_pixels  = np.sum(~np.isnan(ndvi_arr))
forest_pixels = np.sum(forest_mask)
forest_pct    = 100 * forest_pixels / total_pixels

print(f'Total valid pixels : {total_pixels:,}')
print(f'Forest pixels      : {forest_pixels:,}  ({forest_pct:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
p2, p98 = np.nanpercentile(ndvi_arr, [2, 98])
axes[0].imshow(ndvi_arr, cmap='RdYlGn', vmin=p2, vmax=p98, interpolation='bilinear')
axes[0].set_title('NDVI', fontsize=12); axes[0].axis('off')

mask_vis = np.where(forest_mask, 1.0, np.where(np.isnan(ndvi_arr), np.nan, 0.0))
axes[1].imshow(mask_vis, cmap='Greens', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_title(f'Forest Mask (NDVI > {ndvi_thresh}) — {forest_pct:.1f}% of pixels', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()